In [1]:
!git clone https://github.com/Team-TUD/CTAB-GAN-Plus
import sys
sys.path.append('./CTAB-GAN-Plus')
from model.ctabgan import CTABGAN

fatal: destination path 'CTAB-GAN-Plus' already exists and is not an empty directory.


In [2]:
from ucimlrepo import fetch_ucirepo
import pandas as pd
import numpy as np
import random
import torch
import torch.nn as nn
import torch.optim as optim

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

from sdv.metadata import SingleTableMetadata
from sdv.single_table import (
    CTGANSynthesizer,
    CopulaGANSynthesizer,
    TVAESynthesizer,
    GaussianCopulaSynthesizer
)

from sdv.evaluation.single_table import evaluate_quality

from model.ctabgan import CTABGAN

# ----------------------------------------------------
# Load Dataset
# ----------------------------------------------------
covertype = fetch_ucirepo(id=31)

# data (as pandas dataframes)
X = covertype.data.features
y = covertype.data.targets

# metadata
print(covertype.metadata)

# variable information
print(covertype.variables)

covertype_data = pd.concat([X, y], axis=1)

# ----------------------------------------------------
# Target column handling
# ----------------------------------------------------
# Ensure Cover_Type is the explicit target variable
if "Cover_Type" in covertype_data.columns:
    target_col = "Cover_Type"
elif "cover_type" in covertype_data.columns:
    covertype_data = covertype_data.rename(columns={"cover_type": "Cover_Type"})
    target_col = "Cover_Type"
else:
    target_col = covertype_data.columns[-1]
    covertype_data = covertype_data.rename(columns={target_col: "Cover_Type"})
    target_col = "Cover_Type"

# Keep full dataset for repeatable subsampling when experiment settings is re-run.
covertype_data_full = covertype_data.copy()
print(f"Full dataset cached: {covertype_data_full.shape}")


{'uci_id': 31, 'name': 'Covertype', 'repository_url': 'https://archive.ics.uci.edu/dataset/31/covertype', 'data_url': 'https://archive.ics.uci.edu/static/public/31/data.csv', 'abstract': 'Classification of pixels into 7 forest cover types based on attributes such as elevation, aspect, slope, hillshade, soil-type, and more.', 'area': 'Biology', 'tasks': ['Classification'], 'characteristics': ['Multivariate'], 'num_instances': 581012, 'num_features': 54, 'feature_types': ['Categorical', 'Integer'], 'demographics': [], 'target_col': ['Cover_Type'], 'index_col': None, 'has_missing_values': 'no', 'missing_values_symbol': None, 'year_of_dataset_creation': 1998, 'last_updated': 'Sat Mar 16 2024', 'dataset_doi': '10.24432/C50K5N', 'creators': ['Jock Blackard'], 'intro_paper': None, 'additional_info': {'summary': 'Predicting forest cover type from cartographic variables only (no remotely sensed data).  The actual forest cover type for a given observation (30 x 30 meter cell) was determined from

In [3]:
# ----------------------------------------------------
# Preprocess features before synthetic data generation
# ----------------------------------------------------

X = covertype_data_full.drop(columns=[target_col]).copy()
y = covertype_data_full[target_col].copy()

covertype_data_full = pd.concat([X, y], axis=1)
covertype_data = covertype_data_full.copy()

print(f'Target variable: {target_col}')
print(f'Dataset shape after preprocessing: {covertype_data_full.shape}')


Target variable: Cover_Type
Dataset shape after preprocessing: (581012, 55)


In [4]:
# ----------------------------------------------------
# Experiment Settings
# ----------------------------------------------------
N_SAMPLES = 1000          # random real samples drawn from full dataset
TEST_SIZE = 0.2           # 20% holdout for unseen TSTR evaluation
SEED = 42

# Speed controls (set DEV_MODE=False, FAST_MODE=False for full paper run)
FAST_MODE = True
DEV_MODE = True
RUN_QUALITY_EVAL = True

N_SYNTH_SAMPLES = 200 if DEV_MODE else 1000

_epoch_fast = 2 if DEV_MODE else 5
CTABGAN_EPOCHS = 50
WGAN_EPOCHS = 50 if FAST_MODE else 100
SDV_EPOCHS = _epoch_fast if FAST_MODE else 300

EVAL_SEEDS = [42, 43, 44, 45, 46, 47, 48, 49, 50, 51]

ALL_GENERATORS = [
    "CTGAN", "CopulaGAN", "TVAE", "GaussianCopula", "WGAN_GP", "CTABGAN"
]
GENERATORS_TO_EVAL = ALL_GENERATORS

# All 6 generators use exactly these 10 cartographic features + Cover_Type target.
CONTINUOUS_COLS = [
    "Elevation",
    "Aspect",
    "Slope",
    "Horizontal_Distance_To_Hydrology",
    "Vertical_Distance_To_Hydrology",
    "Horizontal_Distance_To_Roadways",
    "Hillshade_9am",
    "Hillshade_Noon",
    "Hillshade_3pm",
    "Horizontal_Distance_To_Fire_Points",
]

GENERATOR_FEATURE_COLS = CONTINUOUS_COLS.copy()
if len(covertype_data_full) < N_SAMPLES:
    raise RuntimeError(
        f"Full dataset has only {len(covertype_data_full)} rows. "
        "Re-run the data loading and preprocessing cells first."
    )

covertype_data, _ = train_test_split(
    covertype_data_full,
    train_size=N_SAMPLES,
    stratify=covertype_data_full[target_col],
    random_state=SEED,
)
covertype_data = covertype_data.reset_index(drop=True)

# 80% for generator training, 20% held out unseen for TSTR evaluation.
train_real, test_real = train_test_split(
    covertype_data,
    test_size=TEST_SIZE,
    stratify=covertype_data[target_col],
    random_state=SEED,
)
train_real = train_real.reset_index(drop=True)
test_real = test_real.reset_index(drop=True)

metadata = SingleTableMetadata()
metadata.detect_from_dataframe(train_real)

GENERATOR_FEATURE_COLS = [c for c in GENERATOR_FEATURE_COLS if c in train_real.columns]
if len(GENERATOR_FEATURE_COLS) != 10:
    raise ValueError(
        f"Expected 10 shared generator features, found {len(GENERATOR_FEATURE_COLS)}: "
        f"{GENERATOR_FEATURE_COLS}"
    )

GEN_COLS = GENERATOR_FEATURE_COLS + [target_col]
train_gen = train_real[GEN_COLS].copy()
test_gen = test_real[GEN_COLS].copy()

train_metadata = SingleTableMetadata()
train_metadata.detect_from_dataframe(train_gen)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

scores = {}
synthetic_datasets = {}
quality_results = []

print(f'Random subsample: {covertype_data.shape}')
print(f'Generator training set (shared 10 features): {train_gen.shape}')
print(f'Holdout test set (shared 10 features): {test_gen.shape}')
print(f'DEV_MODE: {DEV_MODE} | FAST_MODE: {FAST_MODE} | quality eval: {RUN_QUALITY_EVAL}')
print(f'Generators enabled: {GENERATORS_TO_EVAL}')
print(f'Synthetic samples per generator: {N_SYNTH_SAMPLES}')
print(f'All 6 generators use these 10 features + target: {GEN_COLS}')


Random subsample: (1000, 55)
Generator training set (shared 10 features): (800, 11)
Holdout test set (shared 10 features): (200, 11)
DEV_MODE: True | FAST_MODE: True | quality eval: True
Generators enabled: ['CTGAN', 'CopulaGAN', 'TVAE', 'GaussianCopula', 'WGAN_GP', 'CTABGAN']
Synthetic samples per generator: 200
All 6 generators use these 10 features + target: ['Elevation', 'Aspect', 'Slope', 'Horizontal_Distance_To_Hydrology', 'Vertical_Distance_To_Hydrology', 'Horizontal_Distance_To_Roadways', 'Hillshade_9am', 'Hillshade_Noon', 'Hillshade_3pm', 'Horizontal_Distance_To_Fire_Points', 'Cover_Type']


In [5]:
# ---------------------------------------------------
# SINGLE RUN — setup + CTABGAN
# ---------------------------------------------------
seed = SEED

print("\n================ SINGLE RUN ================")

np.random.seed(seed)
random.seed(seed)
torch.manual_seed(seed)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)

# CTABGAN uses the same shared feature set as the other generators.
GENERATOR_INTEGER = [col for col in GENERATOR_FEATURE_COLS]


def prepare_ctabgan_train_df(df, label_col, integer_cols):
    """Cast target to string (categorical) and numeric features to float for CTAB-GAN+."""
    out = df.copy()
    out[label_col] = out[label_col].astype(str)
    for col in integer_cols:
        out[col] = pd.to_numeric(out[col], errors="coerce").fillna(0).astype(np.float64)
    return out


def sanitize_ctabgan_sample(raw, data_prep, ref_df, integer_cols):
    """Replace NaN/inf in generator output before inverse_prep integer casting."""
    df = pd.DataFrame(raw, columns=data_prep.df.columns)
    rng = np.random.default_rng(SEED)

    for enc in getattr(data_prep, "label_encoder_list", []):
        col = enc["column"]
        n_classes = len(enc["label_encoder"].classes_)
        vals = pd.to_numeric(df[col], errors="coerce").replace([np.inf, -np.inf], np.nan)
        vals = vals.fillna(0).clip(0, n_classes - 1)
        df[col] = np.round(vals)

    for col in integer_cols:
        if col not in df.columns or col not in ref_df.columns:
            continue
        ref = pd.to_numeric(ref_df[col], errors="coerce").replace([np.inf, -np.inf], np.nan).dropna()
        if ref.empty:
            continue
        vals = pd.to_numeric(df[col], errors="coerce").replace([np.inf, -np.inf], np.nan)
        if vals.notna().sum() == 0:
            df[col] = rng.choice(ref.values, size=len(df))
        else:
            vals = vals.fillna(ref.median()).clip(ref.min(), ref.max())
            df[col] = vals

    return df.to_numpy()


def align_synthetic_to_real(synth_df, real_df, label_col):
    """Match synthetic column dtypes to the real training frame for SDV quality eval."""
    out = synth_df.copy()
    for col in real_df.columns:
        if col not in out.columns:
            continue
        if col == label_col:
            out[col] = pd.to_numeric(out[col], errors="coerce").fillna(real_df[col].mode()[0])
            out[col] = (
                out[col]
                .round()
                .clip(real_df[col].min(), real_df[col].max())
                .astype(int)
            )
        else:
            out[col] = pd.to_numeric(out[col], errors="coerce").fillna(real_df[col].median())
            if pd.api.types.is_integer_dtype(real_df[col]):
                out[col] = (
                    out[col]
                    .round()
                    .clip(real_df[col].min(), real_df[col].max())
                    .astype(int)
                )
    return out[real_df.columns]


def fast_ctabgan_sample(synthesizer, n, max_resample_rounds=8):
    """Cap CTAB-GAN+ rejection loop (default sample() can run for hours on wide data)."""
    import torch
    from model.synthesizer.ctabgan_synthesizer import apply_activate

    synthesizer.generator.eval()
    output_info = synthesizer.transformer.output_info
    batch_size = synthesizer.batch_size

    def _generate_batch(count):
        chunks = []
        steps = max(1, count // batch_size + 1)
        for _ in range(steps):
            noisez = torch.randn(batch_size, synthesizer.random_dim, device=synthesizer.device)
            condvec = synthesizer.cond_generator.sample(batch_size)
            c = torch.from_numpy(condvec).to(synthesizer.device)
            noisez = torch.cat([noisez, c], dim=1)
            noisez = noisez.view(
                batch_size,
                synthesizer.random_dim + synthesizer.cond_generator.n_opt,
                1,
                1,
            )
            fake = synthesizer.generator(noisez)
            faket = synthesizer.Gtransformer.inverse_transform(fake)
            fakeact = apply_activate(faket, output_info)
            chunks.append(fakeact.detach().cpu().numpy())
        return np.concatenate(chunks, axis=0)

    result, resample = synthesizer.transformer.inverse_transform(_generate_batch(n))
    rounds = 0
    while len(result) < n and rounds < max_resample_rounds:
        rounds += 1
        extra, resample = synthesizer.transformer.inverse_transform(
            _generate_batch(max(resample, batch_size))
        )
        if len(extra):
            result = np.concatenate([result, extra], axis=0)
        print(f"  CTABGAN sample round {rounds}: {len(result)}/{n} valid rows", flush=True)

    if len(result) == 0:
        raise RuntimeError(
            "CTABGAN produced zero valid rows."
        )

    if len(result) < n:
        idx = np.random.choice(len(result), size=n, replace=True)
        result = result[idx]

    return result[:n]


# CTABGAN
if "CTABGAN" in GENERATORS_TO_EVAL:
    import traceback

    try:
        data_path = "covertype_train.csv"
        ctab_train = prepare_ctabgan_train_df(train_gen, target_col, GENERATOR_INTEGER)
        ctab_train.to_csv(data_path, index=False)

        print(f"CTABGAN: training on {len(GENERATOR_FEATURE_COLS)} shared features")

        ctabgan = CTABGAN(
            raw_csv_path=data_path,
            categorical_columns=[target_col],
            log_columns=[],
            mixed_columns={},
            general_columns=GENERATOR_INTEGER,
            integer_columns=GENERATOR_INTEGER,
            problem_type={"Classification": target_col},
        )

        ctabgan.synthesizer.epochs = CTABGAN_EPOCHS
        ctabgan.fit()

        print(f"CTABGAN: sampling {N_SYNTH_SAMPLES} synthetic rows...")
        raw_sample = fast_ctabgan_sample(ctabgan.synthesizer, N_SYNTH_SAMPLES)
        raw_sample = sanitize_ctabgan_sample(
            raw_sample, ctabgan.data_prep, ctab_train, GENERATOR_INTEGER
        )
        synthetic_ctabgan = ctabgan.data_prep.inverse_prep(raw_sample)
        synthetic_ctabgan = align_synthetic_to_real(
            synthetic_ctabgan, train_gen, target_col
        )
        print(
            f"CTABGAN synth: shape={synthetic_ctabgan.shape}, "
            f"target nunique={synthetic_ctabgan[target_col].nunique()}"
        )

        synthetic_datasets["CTABGAN"] = synthetic_ctabgan.copy()

        if RUN_QUALITY_EVAL:
            print("CTABGAN: running SDV quality evaluation...")
            quality = evaluate_quality(
                real_data=train_gen,
                synthetic_data=synthetic_ctabgan,
                metadata=train_metadata,
            )
            score = quality.get_score()
            scores["CTABGAN"] = score
            print("CTABGAN:", round(score, 4))
        else:
            print("CTABGAN: trained (quality eval skipped)")

    except Exception as e:
        print("CTABGAN Failed:", e)
        traceback.print_exc()
else:
    print("CTABGAN: skipped (not in GENERATORS_TO_EVAL)")


================ SINGLE RUN ================
CTABGAN: training on 10 shared features


100%|██████████| 50/50 [01:11<00:00,  1.42s/it]


Finished training in 72.33066916465759  seconds.
CTABGAN: sampling 200 synthetic rows...
CTABGAN synth: shape=(200, 11), target nunique=7
CTABGAN: running SDV quality evaluation...
Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 11/11 [00:00<00:00, 341.22it/s]|
Column Shapes Score: 75.18%

(2/2) Evaluating Column Pair Trends: |██████████| 55/55 [00:00<00:00, 99.35it/s]| 
Column Pair Trends Score: 83.92%

Overall Score (Average): 79.55%

CTABGAN: 0.7955


In [6]:
# ---------------------------------------------------
# WGAN-GP
# ---------------------------------------------------
if "WGAN_GP" in GENERATORS_TO_EVAL:
    try:
        import traceback
        from sklearn.preprocessing import StandardScaler

        def finalize_wgan_synthetic(synth_df, ref_df, label_col, encoder, feature_cols):
            """Clip features and ensure multi-class target for downstream TSTR."""
            out = synth_df.copy()
            valid_classes = np.array(sorted(pd.to_numeric(ref_df[label_col], errors="coerce").dropna().astype(int).unique()))

            enc_vals = (
                out[label_col]
                .round()
                .clip(0, len(encoder.classes_) - 1)
                .astype(int)
            )
            out[label_col] = encoder.inverse_transform(enc_vals).astype(int)

            def _resample_target_from_real(n_rows):
                y = ref_df[label_col]
                if y.nunique() >= 2 and y.value_counts().min() >= 2 and n_rows <= len(y):
                    sampled, _ = train_test_split(
                        y,
                        train_size=n_rows,
                        stratify=y,
                        random_state=SEED,
                    )
                    return sampled.reset_index(drop=True).astype(int).values
                return (
                    y.sample(n=n_rows, replace=True, random_state=SEED)
                    .reset_index(drop=True)
                    .astype(int)
                    .values
                )

            if out[label_col].nunique() < 2 or out[label_col].value_counts().min() < 2:
                out[label_col] = _resample_target_from_real(len(out))
            else:
                out[label_col] = out[label_col].apply(
                    lambda v: valid_classes[np.argmin(np.abs(valid_classes - int(v)))]
                ).astype(int)
                if out[label_col].nunique() < 2 or out[label_col].value_counts().min() < 2:
                    out[label_col] = _resample_target_from_real(len(out))

            for col in feature_cols:
                col_min = int(ref_df[col].min())
                col_max = int(ref_df[col].max())
                out[col] = (
                    out[col]
                    .round()
                    .clip(col_min, col_max)
                    .astype(int)
                )

            return out

        data_wgan = train_gen.copy()

        encoder = LabelEncoder()
        data_wgan[target_col] = encoder.fit_transform(data_wgan[target_col])

        scaler = StandardScaler()
        scaled_values = scaler.fit_transform(data_wgan.values.astype(np.float32))

        device = "cuda" if torch.cuda.is_available() else "cpu"

        real_tensor = torch.tensor(
            scaled_values,
            dtype=torch.float32
        )

        batch_size = 64
        latent_dim = 64
        data_dim = real_tensor.shape[1]

        loader = torch.utils.data.DataLoader(
            real_tensor,
            batch_size=batch_size,
            shuffle=True,
            drop_last=False
        )

        class Generator(nn.Module):
            def __init__(self):
                super().__init__()

                self.model = nn.Sequential(
                    nn.Linear(latent_dim, 128),
                    nn.LayerNorm(128),
                    nn.LeakyReLU(0.2),

                    nn.Linear(128, 256),
                    nn.LayerNorm(256),
                    nn.LeakyReLU(0.2),

                    nn.Linear(256, data_dim)
                )

            def forward(self, z):
                return self.model(z)

        class Critic(nn.Module):
            def __init__(self):
                super().__init__()

                self.model = nn.Sequential(
                    nn.Linear(data_dim, 256),
                    nn.LeakyReLU(0.2),

                    nn.Linear(256, 128),
                    nn.LeakyReLU(0.2),

                    nn.Linear(128, 1)
                )

            def forward(self, x):
                return self.model(x)

        generator = Generator().to(device)
        critic = Critic().to(device)

        optimizer_G = optim.Adam(
            generator.parameters(),
            lr=0.0001,
            betas=(0.5, 0.9)
        )

        optimizer_C = optim.Adam(
            critic.parameters(),
            lr=0.0001,
            betas=(0.5, 0.9)
        )

        def gradient_penalty(critic, real_samples, fake_samples):

            alpha = torch.rand(real_samples.size(0), 1, device=device)
            alpha = alpha.expand_as(real_samples)

            interpolates = (
                alpha * real_samples +
                (1 - alpha) * fake_samples
            ).requires_grad_(True)

            critic_interpolates = critic(interpolates)

            gradients = torch.autograd.grad(
                outputs=critic_interpolates,
                inputs=interpolates,
                grad_outputs=torch.ones_like(critic_interpolates),
                create_graph=True,
                retain_graph=True
            )[0]

            gradients = gradients.view(gradients.size(0), -1)

            return ((gradients.norm(2, dim=1) - 1) ** 2).mean()

        for epoch in range(WGAN_EPOCHS):

            for real_batch in loader:

                real_batch = real_batch.to(device)

                for _ in range(5):

                    z = torch.randn(
                        real_batch.size(0),
                        latent_dim,
                        device=device
                    )

                    fake_batch = generator(z).detach()

                    critic_real = critic(real_batch).mean()
                    critic_fake = critic(fake_batch).mean()

                    gp = gradient_penalty(
                        critic,
                        real_batch,
                        fake_batch
                    )

                    critic_loss = (
                        critic_fake
                        - critic_real
                        + 10 * gp
                    )

                    optimizer_C.zero_grad()
                    critic_loss.backward()
                    optimizer_C.step()

                z = torch.randn(
                    real_batch.size(0),
                    latent_dim,
                    device=device
                )

                fake = generator(z)

                generator_loss = -critic(fake).mean()

                optimizer_G.zero_grad()
                generator_loss.backward()
                optimizer_G.step()

        generator.eval()

        with torch.no_grad():

            z = torch.randn(
                N_SYNTH_SAMPLES,
                latent_dim,
                device=device
            )

            synthetic_scaled = generator(z).cpu().numpy()

        synthetic_raw = scaler.inverse_transform(synthetic_scaled)
        synthetic_wgan = pd.DataFrame(
            synthetic_raw,
            columns=data_wgan.columns
        )

        synthetic_wgan = finalize_wgan_synthetic(
            synthetic_wgan,
            train_gen,
            target_col,
            encoder,
            GENERATOR_FEATURE_COLS,
        )

        print(
            f"WGAN_GP synth: shape={synthetic_wgan.shape}, "
            f"target classes={sorted(synthetic_wgan[target_col].unique())}"
        )

        synthetic_datasets["WGAN_GP"] = synthetic_wgan.copy()

        if RUN_QUALITY_EVAL:
            quality = evaluate_quality(
                real_data=train_gen,
                synthetic_data=synthetic_wgan,
                metadata=train_metadata
            )
            scores["WGAN_GP"] = quality.get_score()
            print("WGAN_GP:", round(scores["WGAN_GP"], 4))
        else:
            print("WGAN_GP: trained (quality eval skipped)")

        del generator
        del critic
        del real_tensor

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    except Exception as e:
        print("WGAN_GP Failed:")
        traceback.print_exc()
else:
    print("WGAN_GP: skipped (not in GENERATORS_TO_EVAL)")


WGAN_GP synth: shape=(200, 11), target classes=[1, 2, 3, 4, 5, 6, 7]
Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 11/11 [00:00<00:00, 645.77it/s]|
Column Shapes Score: 83.58%

(2/2) Evaluating Column Pair Trends: |██████████| 55/55 [00:00<00:00, 194.98it/s]|
Column Pair Trends Score: 88.93%

Overall Score (Average): 86.26%

WGAN_GP: 0.8626


In [7]:
# ---------------------------------------------------
# SDV Models (CTGAN, CopulaGAN, TVAE, GaussianCopula)
# ---------------------------------------------------
sdv_models = {
    "CTGAN": CTGANSynthesizer(metadata=train_metadata, epochs=SDV_EPOCHS),
    "CopulaGAN": CopulaGANSynthesizer(metadata=train_metadata, epochs=SDV_EPOCHS),
    "TVAE": TVAESynthesizer(metadata=train_metadata, epochs=SDV_EPOCHS),
    "GaussianCopula": GaussianCopulaSynthesizer(metadata=train_metadata),
}

for model_name, model in sdv_models.items():
    if model_name not in GENERATORS_TO_EVAL:
        print(f"{model_name}: skipped (not in GENERATORS_TO_EVAL)")
        continue

    try:
        model.fit(train_gen)
        synthetic_data = model.sample(N_SYNTH_SAMPLES)
        synthetic_datasets[model_name] = synthetic_data.copy()

        if RUN_QUALITY_EVAL:
            quality = evaluate_quality(
                real_data=train_gen,
                synthetic_data=synthetic_data,
                metadata=train_metadata
            )
            scores[model_name] = quality.get_score()
            print(f"{model_name}: {round(scores[model_name], 4)}")
        else:
            print(f"{model_name}: trained (quality eval skipped)")

    except Exception as e:
        print(f"{model_name} Failed: {e}")


Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 11/11 [00:00<00:00, 414.39it/s]|
Column Shapes Score: 73.01%

(2/2) Evaluating Column Pair Trends: |██████████| 55/55 [00:00<00:00, 182.44it/s]|
Column Pair Trends Score: 79.81%

Overall Score (Average): 76.41%

CTGAN: 0.7641
Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 11/11 [00:00<00:00, 281.82it/s]|
Column Shapes Score: 69.84%

(2/2) Evaluating Column Pair Trends: |██████████| 55/55 [00:00<00:00, 195.46it/s]|
Column Pair Trends Score: 78.47%

Overall Score (Average): 74.15%

CopulaGAN: 0.7415
Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 11/11 [00:00<00:00, 367.79it/s]|
Column Shapes Score: 65.5%

(2/2) Evaluating Column Pair Trends: |██████████| 55/55 [00:00<00:00, 167.59it/s]|
Column Pair Trends Score: 82.16%

Overall Score (Average): 73.83%

TVAE: 0.7383
Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 11/11 [00:00<00:00, 608.03it/s]|
Column Shapes S

In [8]:
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC, LinearSVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import (
    RandomForestClassifier,
    GradientBoostingClassifier,
    AdaBoostClassifier,
    ExtraTreesClassifier,
)
from sklearn.tree import DecisionTreeClassifier
from sklearn.neural_network import MLPClassifier

EVAL_SEEDS = [42, 43, 44, 45, 46, 47, 48, 49, 50, 51]

if FAST_MODE:
    models = {
        "LogReg": LogisticRegression(max_iter=500, solver="liblinear", random_state=42),
        "SVM-RBF": LinearSVC(max_iter=500, dual="auto", random_state=42),
        "KNN": KNeighborsClassifier(n_neighbors=5, n_jobs=-1),
        "NaiveBayes": GaussianNB(),
        "DecisionTree": DecisionTreeClassifier(random_state=42, max_depth=12),
        "RandomForest": RandomForestClassifier(
            n_estimators=50, random_state=42, n_jobs=-1
        ),
        "ExtraTrees": ExtraTreesClassifier(
            n_estimators=50, random_state=42, n_jobs=-1
        ),
        "GradientBoost": GradientBoostingClassifier(
            n_estimators=30, random_state=42
        ),
        "AdaBoost": AdaBoostClassifier(n_estimators=30, random_state=42),
        "MLP": MLPClassifier(max_iter=200, random_state=42),
    }
else:
    models = {
        "LogReg": LogisticRegression(max_iter=5000, solver="liblinear", random_state=42),
        "SVM-RBF": SVC(kernel="rbf", cache_size=1000, tol=1e-3, random_state=42),
        "KNN": KNeighborsClassifier(n_jobs=-1),
        "NaiveBayes": GaussianNB(),
        "DecisionTree": DecisionTreeClassifier(random_state=42),
        "RandomForest": RandomForestClassifier(random_state=42, n_jobs=-1),
        "ExtraTrees": ExtraTreesClassifier(random_state=42, n_jobs=-1),
        "GradientBoost": GradientBoostingClassifier(random_state=42),
        "AdaBoost": AdaBoostClassifier(random_state=42),
        "MLP": MLPClassifier(max_iter=500, random_state=42),
    }

print(f"Classifier evaluation: {len(models)} models, {len(EVAL_SEEDS)} seeds")
if FAST_MODE:
    print("FAST_MODE: SVM-RBF uses LinearSVC (linear kernel) for speed.")


Classifier evaluation: 10 models, 10 seeds
FAST_MODE: SVM-RBF uses LinearSVC (linear kernel) for speed.


In [9]:
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from sklearn.base import clone
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
import numpy as np
import pandas as pd


In [10]:
# TRTR is evaluated in the comparison cell below via evaluate_models().
print(
    "Skipping duplicate TRTR cell. "
    f"Run the comparison cell for TRTR/TSTR ({len(models)} models, {len(EVAL_SEEDS)} seeds)."
)


Skipping duplicate TRTR cell. Run the comparison cell for TRTR/TSTR (10 models, 10 seeds).


In [11]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from sklearn.base import clone
import pandas as pd
import numpy as np


def _safe_stratify(y):
    y = pd.Series(y).reset_index(drop=True)
    if y.nunique() < 2 or y.value_counts().min() < 2:
        return None
    return y


def evaluate_models(
    train_df,
    test_df,
    label_col,
    models,
    test_size=0.2,
    seeds=None,
    use_holdout=False,
):
    if seeds is None:
        seeds = EVAL_SEEDS

    def _std(values):
        return float(np.std(values, ddof=1)) if len(values) > 1 else 0.0

    results = []

    for name, model in models.items():
        print(f"  {name}...", flush=True)

        accuracy_scores = []
        f1_scores = []
        precision_scores = []
        recall_scores = []

        for seed in seeds:
            X_train_full = train_df.drop(columns=[label_col])
            y_train_full = train_df[label_col]
            X_test = test_df.drop(columns=[label_col])
            y_test = test_df[label_col]

            if use_holdout:
                # Fixed holdout test; resample training rows per seed.
                X_train, _, y_train, _ = train_test_split(
                    X_train_full,
                    y_train_full,
                    test_size=test_size,
                    random_state=seed,
                    stratify=_safe_stratify(y_train_full),
                )
            else:
                X_train, _, y_train, _ = train_test_split(
                    X_train_full,
                    y_train_full,
                    test_size=test_size,
                    random_state=seed,
                    stratify=_safe_stratify(y_train_full),
                )

                _, X_test, _, y_test = train_test_split(
                    X_test,
                    y_test,
                    test_size=test_size,
                    random_state=seed,
                    stratify=_safe_stratify(y_test),
                )

            clf = clone(model)
            if 'random_state' in clf.get_params():
                clf.set_params(random_state=seed)
            if hasattr(clf, "n_jobs"):
                clf.set_params(n_jobs=-1)

            clf.fit(X_train, y_train)
            y_pred = clf.predict(X_test)

            accuracy_scores.append(accuracy_score(y_test, y_pred))
            f1_scores.append(
                f1_score(y_test, y_pred, average="weighted", zero_division=0)
            )
            precision_scores.append(
                precision_score(y_test, y_pred, average="weighted", zero_division=0)
            )
            recall_scores.append(
                recall_score(y_test, y_pred, average="weighted", zero_division=0)
            )

        results.append({
            "Model": name,
            "Accuracy Mean": np.mean(accuracy_scores),
            "Accuracy Std": _std(accuracy_scores),
            "F1 Mean": np.mean(f1_scores),
            "F1 Std": _std(f1_scores),
            "Precision Mean": np.mean(precision_scores),
            "Precision Std": _std(precision_scores),
            "Recall Mean": np.mean(recall_scores),
            "Recall Std": _std(recall_scores),
            "Accuracy (Mean±Std)": f"{np.mean(accuracy_scores):.4f} ± {_std(accuracy_scores):.4f}",
            "F1 (Mean±Std)": f"{np.mean(f1_scores):.4f} ± {_std(f1_scores):.4f}",
            "Precision (Mean±Std)": f"{np.mean(precision_scores):.4f} ± {_std(precision_scores):.4f}",
            "Recall (Mean±Std)": f"{np.mean(recall_scores):.4f} ± {_std(recall_scores):.4f}",
        })

    return pd.DataFrame(results).sort_values(by="Accuracy Mean", ascending=False)


In [12]:
import pandas as pd

label_col = "Cover_Type"

model_order = [
    "CTGAN",
    "CopulaGAN",
    "TVAE",
    "GaussianCopula",
    "WGAN_GP",
    "CTABGAN"
]

seeds = EVAL_SEEDS

print("TRTR (Train Real, Test Real) — 80% train / 20% holdout")
print(
    f"Classifiers: {len(models)} | Seeds: {len(seeds)} | "
    f"Generators: {len(model_order)}"
)
print(f"Generator training set: {train_gen.shape} | Holdout test set: {test_gen.shape}")

trtr_results = evaluate_models(
    train_df=train_gen,
    test_df=test_gen,
    label="Cover_Type",
    models=models,
    seeds=seeds,
    use_holdout=True,
)

display(
    trtr_results[
        [
            "Model",
            "Accuracy (Mean±Std)",
            "F1 (Mean±Std)",
            "Precision (Mean±Std)",
            "Recall (Mean±Std)"
        ]
    ]
)

print("=" * 70)

all_comparisons = []

for synth_name in model_order:

    if synth_name not in synthetic_datasets:
        print(f"Skipping {synth_name} — not in synthetic_datasets")
        continue

    print(f"{synth_name} - TSTR (train on synthetic, test on 20% holdout)")

    synthetic_train_df = synthetic_datasets[synth_name]

    tstr_results = evaluate_models(
        train_df=synthetic_train_df,
        test_df=test_gen,
        label="Cover_Type",
        models=models,
        seeds=seeds,
        use_holdout=True,
    )

    display(
        tstr_results[
            [
                "Model",
                "Accuracy (Mean±Std)",
                "F1 (Mean±Std)",
                "Precision (Mean±Std)",
                "Recall (Mean±Std)"
            ]
        ]
    )

    comparison = trtr_results.merge(
        tstr_results,
        on="Model",
        suffixes=("_TRTR", "_TSTR")
    )

    comparison["Accuracy_Drop"] = (
        comparison["Accuracy Mean_TRTR"]
        - comparison["Accuracy Mean_TSTR"]
    )

    comparison["F1_Drop"] = (
        comparison["F1 Mean_TRTR"]
        - comparison["F1 Mean_TSTR"]
    )

    comparison["Precision_Drop"] = (
        comparison["Precision Mean_TRTR"]
        - comparison["Precision Mean_TSTR"]
    )

    comparison["Recall_Drop"] = (
        comparison["Recall Mean_TRTR"]
        - comparison["Recall Mean_TSTR"]
    )

    comparison["Synthetic_Model"] = synth_name

    print(f"{synth_name} - TRTR vs TSTR")

    display(
        comparison[
            [
                "Synthetic_Model",
                "Model",
                "Accuracy_Drop",
                "F1_Drop",
                "Precision_Drop",
                "Recall_Drop",
                "Accuracy (Mean±Std)_TRTR",
                "Accuracy (Mean±Std)_TSTR"
            ]
        ]
    )

    all_comparisons.append(comparison)

combined_comparison = pd.concat(
    all_comparisons,
    ignore_index=True
)

summary = (
    combined_comparison
    .groupby("Synthetic_Model", as_index=False)
    [["Accuracy_Drop", "F1_Drop", "Precision_Drop", "Recall_Drop"]]
    .mean()
    .sort_values("Accuracy_Drop")
)

print("Average metric drop by synthetic generator (lower is better)")

display(summary)


TRTR (Train Real, Test Real) — 80% train / 20% holdout
Classifiers: 10 | Seeds: 10 | Generators: 6
Generator training set: (800, 11) | Holdout test set: (200, 11)
  LogReg...
  SVM-RBF...
  KNN...
  NaiveBayes...
  DecisionTree...
  RandomForest...
  ExtraTrees...
  GradientBoost...
  AdaBoost...
  MLP...


,Model,Accuracy (Mean±Std),F1 (Mean±Std),Precision (Mean±Std),Recall (Mean±Std)
7,GradientBoost,0.6895 ± 0.0194,0.6818 ± 0.0199,0.6849 ± 0.0241,0.6895 ± 0.0194
6,ExtraTrees,0.6825 ± 0.0189,0.6620 ± 0.0217,0.6587 ± 0.0342,0.6825 ± 0.0189
5,RandomForest,0.6815 ± 0.0190,0.6603 ± 0.0210,0.6583 ± 0.0332,0.6815 ± 0.0190
0,LogReg,0.6560 ± 0.0141,0.6304 ± 0.0151,0.6352 ± 0.0236,0.6560 ± 0.0141
8,AdaBoost,0.6485 ± 0.0278,0.6197 ± 0.0226,0.6169 ± 0.0259,0.6485 ± 0.0278
1,SVM-RBF,0.6405 ± 0.0162,0.6081 ± 0.0180,0.5932 ± 0.0173,0.6405 ± 0.0162
4,DecisionTree,0.6100 ± 0.0231,0.6033 ± 0.0236,0.6061 ± 0.0226,0.6100 ± 0.0231
2,KNN,0.6050 ± 0.0199,0.5859 ± 0.0203,0.5835 ± 0.0316,0.6050 ± 0.0199
3,NaiveBayes,0.5980 ± 0.0187,0.5955 ± 0.0154,0.5978 ± 0.0125,0.5980 ± 0.0187
9,MLP,0.4790 ± 0.0490,0.4506 ± 0.0420,0.5037 ± 0.0460,0.4790 ± 0.0490


CTGAN - TSTR (train on synthetic, test on 20% holdout)
  LogReg...
  SVM-RBF...
  KNN...
  NaiveBayes...
  DecisionTree...
  RandomForest...
  ExtraTrees...
  GradientBoost...
  AdaBoost...
  MLP...


,Model,Accuracy (Mean±Std),F1 (Mean±Std),Precision (Mean±Std),Recall (Mean±Std)
2,KNN,0.1745 ± 0.0203,0.2214 ± 0.0278,0.4076 ± 0.0370,0.1745 ± 0.0203
3,NaiveBayes,0.1680 ± 0.0296,0.1864 ± 0.0240,0.3521 ± 0.0623,0.1680 ± 0.0296
9,MLP,0.1605 ± 0.0689,0.1911 ± 0.0557,0.3925 ± 0.0979,0.1605 ± 0.0689
4,DecisionTree,0.1535 ± 0.0442,0.1987 ± 0.0561,0.4373 ± 0.0667,0.1535 ± 0.0442
7,GradientBoost,0.1475 ± 0.0290,0.1903 ± 0.0366,0.4581 ± 0.0500,0.1475 ± 0.0290
6,ExtraTrees,0.1225 ± 0.0207,0.1449 ± 0.0271,0.3988 ± 0.0761,0.1225 ± 0.0207
5,RandomForest,0.1125 ± 0.0207,0.1415 ± 0.0322,0.4397 ± 0.0773,0.1125 ± 0.0207
0,LogReg,0.1085 ± 0.0280,0.1356 ± 0.0425,0.4694 ± 0.0875,0.1085 ± 0.0280
1,SVM-RBF,0.1045 ± 0.0295,0.1244 ± 0.0430,0.4934 ± 0.1023,0.1045 ± 0.0295
8,AdaBoost,0.0840 ± 0.0297,0.0945 ± 0.0506,0.3545 ± 0.1679,0.0840 ± 0.0297


CTGAN - TRTR vs TSTR


...

,Model,Accuracy (Mean±Std),F1 (Mean±Std),Precision (Mean±Std),Recall (Mean±Std)
8,AdaBoost,0.3775 ± 0.0464,0.3432 ± 0.0271,0.3512 ± 0.0424,0.3775 ± 0.0464
3,NaiveBayes,0.3630 ± 0.0413,0.3997 ± 0.0297,0.4670 ± 0.0156,0.3630 ± 0.0413
1,SVM-RBF,0.3610 ± 0.0237,0.3224 ± 0.0186,0.3248 ± 0.0363,0.3610 ± 0.0237
6,ExtraTrees,0.3570 ± 0.0259,0.3486 ± 0.0222,0.3446 ± 0.0224,0.3570 ± 0.0259
2,KNN,0.3455 ± 0.0210,0.3230 ± 0.0180,0.3385 ± 0.0215,0.3455 ± 0.0210
5,RandomForest,0.3385 ± 0.0299,0.3299 ± 0.0213,0.3262 ± 0.0156,0.3385 ± 0.0299
0,LogReg,0.3300 ± 0.0290,0.3100 ± 0.0196,0.3242 ± 0.0314,0.3300 ± 0.0290
9,MLP,0.2805 ± 0.0513,0.2969 ± 0.0472,0.3814 ± 0.0500,0.2805 ± 0.0513
4,DecisionTree,0.2345 ± 0.0412,0.2691 ± 0.0370,0.3311 ± 0.0389,0.2345 ± 0.0412
7,GradientBoost,0.2285 ± 0.0405,0.2592 ± 0.0349,0.3337 ± 0.0292,0.2285 ± 0.0405


CTABGAN - TRTR vs TSTR


,Synthetic_Model,Model,Accuracy_Drop,F1_Drop,Precision_Drop,Recall_Drop,Accuracy (Mean±Std)_TRTR,Accuracy (Mean±Std)_TSTR
0,CTABGAN,GradientBoost,0.4610,0.422619,0.351205,0.4610,0.6895 ± 0.0194,0.2285 ± 0.0405
1,CTABGAN,ExtraTrees,0.3255,0.313397,0.314100,0.3255,0.6825 ± 0.0189,0.3570 ± 0.0259
2,CTABGAN,RandomForest,0.3430,0.330407,0.332022,0.3430,0.6815 ± 0.0190,0.3385 ± 0.0299
3,CTABGAN,LogReg,0.3260,0.320343,0.311005,0.3260,0.6560 ± 0.0141,0.3300 ± 0.0290
4,CTABGAN,AdaBoost,0.2710,0.276539,0.265755,0.2710,0.6485 ± 0.0278,0.3775 ± 0.0464
5,CTABGAN,SVM-RBF,0.2795,0.285668,0.268400,0.2795,0.6405 ± 0.0162,0.3610 ± 0.0237
6,CTABGAN,DecisionTree,0.3755,0.334244,0.274972,0.3755,0.6100 ± 0.0231,0.2345 ± 0.0412
7,CTABGAN,KNN,0.2595,0.262829,0.244948,0.2595,0.6050 ± 0.0199,0.3455 ± 0.0210
8,CTABGAN,NaiveBayes,0.2350,0.195806,0.130846,0.2350,0.5980 ± 0.0187,0.3630 ± 0.0413
9,CTABGAN,MLP,0.1985,0.153655,0.122371,0.1985,0.4790 ± 0.0490,0.2805 ± 0.0513


Average metric drop by synthetic generator (lower is better)


,Synthetic_Model,Accuracy_Drop,F1_Drop,Precision_Drop,Recall_Drop
3,GaussianCopula,0.16185,0.187001,0.198489,0.16185
5,WGAN_GP,0.22180,0.233687,0.248146,0.22180
4,TVAE,0.24640,0.255781,0.244165,0.24640
0,CTABGAN,0.30745,0.289551,0.261563,0.30745
1,CTGAN,0.49545,0.446885,0.193484,0.49545
2,CopulaGAN,0.53760,0.496370,0.314010,0.53760


In [13]:
output_file = "TRTR_TSTR_results.xlsx"

with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
    
    trtr_results.to_excel(
        writer,
        sheet_name="TRTR_Results",
        index=False
    )

    combined_comparison.to_excel(
        writer,
        sheet_name="All_Comparisons",
        index=False
    )

    summary.to_excel(
        writer,
        sheet_name="Summary",
        index=False
    )

    for synth_name in model_order:
        synth_results = combined_comparison[
            combined_comparison["Synthetic_Model"] == synth_name
        ]

        synth_results.to_excel(
            writer,
            sheet_name=synth_name[:31],
            index=False
        )

print(f"Results saved to: {output_file}")


Results saved to: TRTR_TSTR_results.xlsx
